# 20.18. MP-01 — сопоставление масок MONAI Physio и TotalSegmentator v2

**Статус:** полный wrapper на одной фазе Nix завершил `total`, `heartchambers_highres`, `tissue_4_types`, `lung_vessels`, `body` и сохранил fused labelmap локально. Отдельный частичный сердечный контроль также выполнен и помечен `full_monai_physio_workflow=false`. Ни одно из сравнений не является независимой анатомической валидацией.

**Вопрос.** Меняет ли обёртка MONAI Physio выход TotalSegmentator при одинаковом КТ, задаче, версии весов и параметрах? Согласие двух автоматических масок не является независимой проверкой анатомии.

**Входы.** Указать один псевдоним, одну серию/фазу, дыхательное состояние, один CT NIfTI и бинарные маски одной семантики. Обёртка MONAI Physio смешивает `total`, `heartchambers_highres`, `tissue_4_types`, `lung_vessels` и `body` в единой labelmap. В её выходе label 51 — остаток исходного `total/heart` после наложения других классов; он не эквивалентен сырой маске `total/heart`. Label 140–144 соответствуют highres миокарду и четырём камерам, но их тоже нужно сверить после преобразования координат и обработки. Проверить одинаковые задачу, версию TotalSegmentator и весов, настройку fast/ROI, ориентацию RAS, affine, voxel spacing, FOV, тип интерполяции. Ни один пункт нельзя выводить только из имени файла.

**Решение.** Ненулевой XOR при совпадающих настройках указывает на расхождение маршрутов, которое нужно локализовать. Нулевой XOR означает воспроизводимость конкретного вывода, но не анатомическую валидность. Если входы или версия не установлены, сравнение остаётся диагностическим.

**Наблюдение прямого запуска, 28.09.2026.** ITK не открыл CT по исходному Windows Unicode-пути, но прочитал тот же NIfTI через временный ASCII `subst`-алиас; после запуска алиасы удалены. Первоначальный `WinError 10013` при загрузке Task 485 был sandbox network blocker, не свидетельством недостаточной лицензии. Официальный backend позднее успешно отдал Task 485 при том же локальном академическом ключе. После `tissue_4_types` guard остановил следующий запрос: `lung_vessels` требует публичный Task 117 из официального release TotalSegmentator `v2.5.0-weights`. Один разрешённый повтор с точным GET URL и проверенным asset redirect скачал Task 117 и завершил все пять задач. КТ не отправлялась в сеть. Ключ не записан в локальном журнале.

**Полный выход и одноимённые классы.** Манифест имеет `full_monai_physio_workflow=true`; локальные fused labelmap и отдельные NIfTI сохранены в ignored `runs/`. Для full labels `140–144` против пяти проектных raw `heartchambers_highres` масок совпали shape `512×512×256` и affine; spacing `0,765625×0,765625×0,625`, единица заголовка `unknown`, мм принято явным допущением по происхождению серии. Объёмы union: `569,387` и `571,987` мл; Dice `0,990910`, Jaccard `0,981983`, симметрическая разность `10,375` мл. Это внутреннее согласие автоматических маршрутов с разной предобработкой/ROI; ручной референс и независимая анатомическая точность отсутствуют.

**Межклассовая геометрия полного выхода.** Raw `total/heart` имеет иной смысл, чем highres myocardium+chambers. У полного MONAI union вне raw `total/heart` лежит `16,570` мл (`2,910%` union); пересечение покрывает `88,555%` raw `total/heart`. Это пространственное сопоставление разных классов, не метрика точности или рейтинг. Residual label `51` не сопоставляется с raw whole heart как эквивалент.

**Частичный режим и корректное сравнение.** На той же фазе `set_has_academic_license(True)` сохранён. Диагностический launcher оставляет реальные `total` и `heartchambers_highres`, а `tissue_4_types`, `lung_vessels`, `body` явно заменяет нулевыми ответами; поэтому labels `140–144` сохраняют семантику highres миокарда и четырёх камер, но путь не тождествен полному wrapper. Прямое сравнение partial и full union выявило 10793 различающихся вокселя (между ними Dice 0,996538); вспомогательные проходы и наложения действительно изменили labels `140–144`. Dice partial против проектных raw highres масок 0,994125, full — 0,990910; это не рейтинг анатомической точности. Сравнение ведётся с объединением пяти сырых классов проектного `heartchambers_highres` на общей сетке `512×512×256`, spacing `0,765625×0,765625×0,625` мм при явном допущении мм (`xyzt_units=unknown`), максимальное расхождение affine — 0. Dice объединений `0,994125`, симметрическая разность `6,725` мл, объёмы `571,987` и `572,704` мл. Это внутреннее межзапусковое согласие двух автоматических маршрутов с разной предобработкой, не оценка анатомической точности.

**Межклассовая геометрия.** Raw `total/heart` — иной класс, чем объединение `140–144`. У проектного highres объединения вне raw `total/heart` лежит `15,738` мл (`2,751%` его объёма); у частичного MONAI — `16,501` мл (`2,881%`). Доли raw `total/heart`, покрытые этими объединениями, составляют `89,104%` и `89,097%`. Это контроль пространственного отношения разных классов, не точность сегментации и не ранжирование моделей. Label 51 и лёгочная артерия исключены.


## Решение по старой КТ и полный пофазный QC

[Полный отчёт по старой серии](mp01_old_ct_report.md) охватывает **20 фаз** исторической контрастной КТ и пять одноимённых классов: четыре камеры и миокард. Прямые кандидатные контуры подробно доступны только для фаз 0% и 40%; они были построены после просмотра TS и не являются независимым принятым референсом. Средний Dice между MONAI Physio и проектным TS на 20 фазах лежит от `0,9748` (ПЖ) до `0,9880` (ЛЖ). Наибольшая разница абсолютного объёма ПЖ — `10,41` мл. Это проверка двух автоматических маршрутов одной семьи моделей.

**Решение:** обёртку MONAI можно использовать как техническую альтернативу для исследовательских кандидатных масок; доказанного выигрыша в анатомической точности нет, канонический TS сохранён. После этого допущения полный маршрут выполнен на **всех 12 фазах Ника**. [Пофазный отчёт Ника](mp01_nix_full_cycle_report.md) показывает средний Dice `0,9815–0,9919` по пяти классам, но не независимую анатомическую точность. Из-за отсутствия принятой ручной фазовой разметки автоматические маски не становятся валидированным референсом для FEM.

## Отдельная готовая модель сегментации NVIDIA

[Предобученная NV-Segment-CTMR](mp01_nvsegment_old_report.md) проверена на фазах 0% и 40% старой КТ без обучения на данных проекта. Стандартный режим `CT_BODY` выдал сердце целиком, но не выдал четыре камеры и миокард; явные запросы этих пяти классов технически сработали. Их динамика объёма сильно расходится с TS и прямыми кандидатами, особенно у ЛЖ, ПЖ и ПП. Для количественной динамики камер эта модель **не допущена как замена TS**. Стандартная маска сердца целиком имеет Dice `0,948/0,939` относительно TS на двух фазах; независимой наружной обводки для выбора метода нет.

In [ ]:
import os, subprocess, sys
# Optional local repeat; model download access is controlled by the launcher flags.
# The launcher enforces a fresh child directory inside ignored runs/.
if os.environ.get('MP01_RUN_MONAI') == '1':
    command = [sys.executable, 'mp01_02_run_monai.py', '--ct', os.environ['MP01_CT'], '--output-dir', os.environ.get('MP01_MONAI_OUTPUT', 'runs/mp01_02_monai_candidate')]
    if os.environ.get('MP01_CONTRAST') == '1': command.append('--contrast')
    if os.environ.get('MP01_HEART_ONLY_DIAGNOSTIC') == '1': command.append('--heart-only-diagnostic')
    subprocess.run(command, check=True)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
required = ('MP01_CT', 'MP01_BASELINE_DIR', 'MP01_CANDIDATE_DIR')
missing = [key for key in required if not os.environ.get(key)]
if missing:
    print('Local inputs not configured:', ', '.join(missing))
else:
    script = Path('mp01_02_compare_heart.py')
    output = Path('runs') / f"mp01_02_heart_comparison_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.json"
    command = [sys.executable, str(script), '--ct', os.environ['MP01_CT'], '--baseline-dir', os.environ['MP01_BASELINE_DIR'], '--candidate-dir', os.environ['MP01_CANDIDATE_DIR'], '--output', str(output)]
    if os.environ.get('MP01_WHOLE_HEART'): command.extend(['--whole-heart', os.environ['MP01_WHOLE_HEART']])
    if os.environ.get('MP01_VERIFIED_MM') == '1': command.append('--assume-mm')
    subprocess.run(command, check=True)


**QC/ограничения.** Отчёт `runs/` содержит локальные хэши и affine; он исключён из Git. Перед сравнением сверить CT-хэш, provenance каждой маски и параметры запуска вне этого ноутбука. Если пространственная единица в NIfTI `unknown`, флаг `MP01_VERIFIED_MM=1` допустим только после проверки DICOM/NIfTI происхождения. Ошибка относительно ручного Inobitec-референса остаётся неизвестной без принятого экспорта и соответствия фазы. Дыхательная, сегментационная и межмодельная ошибки учитываются отдельно.
